# Original ApET LLaVA: first execution test

Enable **Internet** and **GPU T4 x2**, preferably in a fresh session.
Push the new files to GitHub first. This notebook runs original ApET in its own
locked Python environment; the notebook kernel stays unchanged.

The first cell uses the `apet-analysis` branch. The test checks the environment,
then generates answers for one red-square image with baseline and combined ApET.
This is an execution smoke test; dataset baselines follow after it works.


In [ ]:
import json
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/doguineyes/CSE590-ApET.git"
REPO_REF = "apet-analysis"  # Change if you pushed a different branch/commit.
repo = Path("/kaggle/working/CSE590-ApET")

if not repo.exists():
    subprocess.run(["git", "clone", REPO_URL, str(repo)], check=True)
subprocess.run(["git", "fetch", "origin"], cwd=repo, check=True)
subprocess.run(["git", "checkout", REPO_REF], cwd=repo, check=True)
if REPO_REF == "apet-analysis":
    subprocess.run(["git", "pull", "--ff-only"], cwd=repo, check=True)
subprocess.run(["git", "rev-parse", "HEAD"], cwd=repo, check=True)


## Install the original environment and check it

No model weights are downloaded here. If it fails, stop and share the output
or the saved JSON report; do not change the notebook kernel's packages.


In [ ]:
runtime_report = Path("/kaggle/working/original-runtime-smoke.json")
subprocess.run(
    ["bash", "bootstrap/setup_original_linux.sh", "--require-cuda",
     "--expected-gpus", "2", "--output", str(runtime_report)],
    cwd=repo, check=True,
)
result = json.loads(runtime_report.read_text())
assert result["status"] == "passed", result
original_python = repo / "environments/original/.venv/bin/python"


## Inspect checkpoint download sizes

The native LLaVA checkpoint and its CLIP vision checkpoint are pinned separately.
They use `/tmp/apet-original-hf-cache`, outside notebook Output. The explicit
20 GiB budget includes both missing checkpoints, with 1 GiB headroom.
This cell queries metadata only. Temporary caches can disappear after sessions.


In [ ]:
preflight_report = Path("/kaggle/working/original-apet-preflight.json")
subprocess.run(
    [str(original_python), "-I", "scripts/smoke_original_apet.py",
     "--preflight-only", "--download-budget-gib", "20",
     "--output", str(preflight_report)],
    cwd=repo, check=True,
)
preflight = json.loads(preflight_report.read_text())
print(json.dumps(preflight["storage"], indent=2))
assert preflight["status"] == "preflight_complete", preflight
assert preflight["storage"]["fits"], "Inspect storage before running inference."


## Run the original baseline and combined compression

A nonempty answer, actual compression counts, and cached generation must pass.
Inspect the content: the expected color is red. No accuracy/speed claim is made.
Combined ApET retains 288 input tokens and reduces to 96 after decoder layer 16.


In [ ]:
smoke_report = Path("/kaggle/working/original-apet-smoke.json")
subprocess.run(
    [str(original_python), "-I", "scripts/smoke_original_apet.py",
     "--download-budget-gib", "20", "--configurations", "baseline", "combined",
     "--output", str(smoke_report)],
    cwd=repo, check=True,
)
smoke = json.loads(smoke_report.read_text())
assert smoke["status"] == "passed", smoke
for run in smoke["runs"]:
    print(run["configuration"], run["response"], run["trace"])


## Download the small result bundle

Download this ZIP from Kaggle Output and share it for review. If an earlier cell
failed, you can still run this cell to package any reports that were written.
For installation failures, share the cell output too.


In [ ]:
from zipfile import ZipFile, ZIP_DEFLATED

bundle = Path("/kaggle/working/original-apet-smoke-results.zip")
with ZipFile(bundle, "w", compression=ZIP_DEFLATED) as archive:
    for name in ["original-runtime-smoke.json", "original-apet-preflight.json",
                 "original-apet-smoke.json", "original-apet-smoke.png"]:
        path = Path("/kaggle/working") / name
        if path.is_file():
            archive.write(path, path.name)
print(bundle)
